In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import ossify as osy

from standard_transform.datasets import v1dd_ds
import statsmodels.api as sm

from caveclient import CAVEclient

# Uncomment if you need to add a token
# CAVEclient.setup_token("https://global.em.brain.allentech.org")

client = CAVEclient("v1dd_public")

## The spatial organization of connectivity

One organizing principles of nervous systems is that neurons use their spatial relationships to build connectivity networks that produce useful functional activity.

Spatial or temporal patterning of excitation and inhibition can create different kinds of activity. For example, excitation followed by inhibition can sharpen responses, shown nicely in [Wehr and Zador 2003](https://pubmed.ncbi.nlm.nih.gov/14647382/) which looked at spikes in auditory cortex while mapping excitatory (green) and inhibitory (red) currents. The individual currents can sum to predict (in yellow) the actual measured voltage response (in navy blue).

![wehr_figure](img/e_i_offset.png){width=500px}

Spatial offsets can create novel circuit computations such as orientation selectivity (e.g. [Rossi, Carandini, and Harris 2020](https://www.nature.com/articles/s41586-020-2894-4)), by having receptive fields for excitation precede that for inhibition only in a prefered direction. This has been well-mapped in neurons of the Drosophila visual system, and a similar result has been found for certain L2/3 pyramidal cells by the Carandini and Harris labs:

![spatial_input](img/spatial_offset_rossi.png){width=500px}

### How is connectivity organized across an axon?

Dense connectomics data is well suited to exploring the relationship between cell types, spatial structure, and synaptic connectivity by giving detailed access to neuronal morphology and connectivity across hundreds or even thousands of synapses.

Here, we will look for evidence of spatial structure in how pyramidal cells distribute their local output across inhibitory and excitatory neurons.
This analysis will put together both skeleton topology and cell type information in order to ask if the type of neurons a pyramidal cell targets depends on the location along the pyramidal cell axon.

This analysis is inspired by a paper from medial entorhinal cortex, a part of the hippocampus, in [Schmidt et al., Nature 2017](https://www.nature.com/articles/nature24005).

This work found an offset between where an excitatory neuron synapsed onto inhibitory neurons (black lines) versus excitatory neurons (magenta lines):

![schmidt_figure_1f](img/axon_sorting.png){width=500px}

We are going to do a similar analysis for pyramidal cells in the V1dd dataset here.

NOTES:
* How might this be implemented by synaptic arbors?
* Go into the biological question posed by axonal sorting.
* Mention the Ribola et al paper for those interested in the dendritic side of this question.
* Great question for the end — if we look at the input side of this, there's also a notion of "electrotonic distance" that carries information about the passive conductance of voltage across a cell. How might we compute the electronic distance of the cell?
* Add a neuroglancer link to see the E and I targets along a cell.

In [ ]:
# To start with, let's build a list of all of the proofread cells and all clearly cell typed cells.

pf_df = client.materialize.tables.proofreading_status_and_strategy(
    status_axon=True
).query(split_positions=True)

ct_df = client.materialize.tables.cell_type_multifeature_v1().get_all(
    split_positions=True, desired_resolution=[1, 1, 1]
)
ct_df = ct_df.drop_duplicates(subset="pt_root_id", keep=False)


# Merge these two into one table and clean the table

pf_ct_df = pf_df.query("status_axon == True").merge(
    ct_df[["id", "pt_root_id", "classification_system", "cell_type"]].rename(
        columns={"id": "cell_id"}
    ),
    on="pt_root_id",
)[
    [
        "pt_root_id",
        "cell_id",
        "classification_system",
        "cell_type",
        "pt_position_x",
        "pt_position_y",
        "pt_position_z",
    ]
]

In [ ]:
root_id = pf_ct_df.query('cell_type == "L3IT"').pt_root_id.iloc[64]

cell = osy.load_cell_from_client(
    root_id=root_id,
    client=client,
    synapses=True,
    include_partner_root_id=True,
)

### Get an idea of what the cell looks like before doing analysis

In [ ]:
osy.plot.plot_morphology_2d(
    cell.transform(v1dd_ds.transform("nm").apply),
    color="compartment",
    palette={osy.SWC_DENDRITE: "black", osy.SWC_AXON: "red"},
    linewidth="radius",
    linewidth_norm=(100, 1000),
    widths=(0.25, 2),
    projection="xy",
)

In [ ]:
# For clarity, let's make sure to only take output synapses on the axon of the cell. This avoids any false detections or small merges elsewhere on the cell.

cell_axon = cell.s.apply_mask(cell.skeleton.df["compartment"] == osy.SWC_AXON)
pre_syn_anno = cell_axon.annotations.pre_syn

# We want to make a normal Pandas dataframe off of the more dynamically generated data in the annotation.

pre_syn_df = pre_syn_anno.df

### What did that mask do?

Ossify masks restrict the cell to only part of the cell.
Because ossify links the skeleton and the synapse annotations, masking on the skeleton also shows only those annotations associated with the skeleton points of the axon.

##### Why did we just do that?

Synapse detection is good, but even well-proofread cells can have errors that can arise from a number of situations. For example,

* Errors in synapse detection or partner assignment add a presynaptic synapse to a dendrite
* A small axonal bouton is merged into a dendrite and was not noticed on proofreading.
* A real biological junction is found on a dendrite and incorrectly identified as a synapse.

By masking out the dendrite, we can avoid some of these some of these errors.

In [ ]:
# Confirm that the masking does what we expected:

osy.plot.plot_morphology_2d(
    cell_axon.transform(v1dd_ds.transform("nm").apply),
    color="compartment",
    palette={osy.SWC_DENDRITE: "black", osy.SWC_AXON: "red"},
    linewidth="radius",
    linewidth_norm=(100, 1000),
    widths=(0.25, 2),
    projection="xy",
)

In [ ]:
pre_syn_df.head()

In [ ]:
pre_syn_df = pre_syn_df.merge(
    ct_df[["pt_root_id", "classification_system", "cell_type"]],
    how="left",
    left_on="post_pt_root_id",
    right_on="pt_root_id",
).rename(
    columns={
        "classification_system": "synapse_target_class"
    }  # Rename the column for clarity
)

In [ ]:
pre_syn_df.head()

In [ ]:
# How many synapses actually have target classifications?

pre_syn_df.value_counts("synapse_target_class")

## Kinds of Distances

There are different ways of measuring the distance between two points in the EM datasets.
Let's consider the distance between a cell body and a synapse (red dot):

![Base neuron image](img/neuron_cartoon_base.png){height=500px}

### Euclidean

The direct line distance between the two points is the Euclidean distance.
It treats all axes the same, which may or may not work well in cortex.

![Euclidean distance](img/neuron_cartoon_euclidean.png){height=500px}

In [ ]:
# Let's check the units

cell.s.root_location

# The high numbers indicate that the location in nm, as is default for skeletons

In [ ]:
euc_distance_um = (
    np.linalg.norm(
        pre_syn_anno.vertices - cell.s.root_location,
        axis=1,
    )
) / 1000.0  # Convert from nm to microns by dividing by 1000

pre_syn_df["dist_to_root_euc"] = euc_distance_um

In [ ]:
sns.histplot(x="dist_to_root_euc", data=pre_syn_df)

### Radial

Another way to measure locations is so-called cylindrical coordinates, where you distinguish a depth axis from a radial distance.
Here, we typically use the y-axis for cortical depth, so this radial distance is specifically within the x-z plane.
This approch treats radial distance as a different measure from distance along depth.

This distance is also called `tangential` or `lateral`.

![Radial](img/neuron_cartoon_cylin.png){height=500px}



#### Streamline-radial distance (the better choice)

In practice, it turns out that cells do not follow a straight line from pia to white matter.
At a given location, apical dendrites and descending axons turn out to follow a "streamline" that is is specific to that location.
When measuring radial distances relative to a cell body at very different layers, it is useful to use the `radial_distance` function in `standard_transform`, which takes this streamline into account.

![streamline](img/steamline_correction_cartoon.png){height=500px}

Note: If you want to straighten a whole cell according to its streamline, you can do that via:
`cell_straightened = cell.transform(v1dd_ds.streamline('nm').transformer(cell.s.root_location))`

In [ ]:
radial_distance_um = v1dd_ds.streamline("nm").radial_distance(
    cell.s.root_location,
    pre_syn_anno.vertices,
)

pre_syn_df["dist_to_root_radial"] = radial_distance_um

### Geodesic Distance

Geodesic distance refers to the shortest path along a surface or graph, and is the same as the **path length** plot in the Schmidt paper.
Here, we use it to refer to the distance along a neuronal skeleton.
Since the skeleton is a tree (i.e. there are no loops), the geodesic distance is the same as the one path between the cell body and the point in question (here, the synapse).

This distance called by other names, including `topological` or `path length`.

![Geodesic](img/neuron_cartoon_geod.png){height=500px}

In [ ]:
# The skeleton is rooted at the cell body. We can simply use the distance_to_root function to get the geodesic distance
# along the skeleton from vertices in an annotation to the root vertex. There is also a pre_syn_anno.distance_between
# to compute a distance between vertices.

pre_syn_df["dist_to_root_geo"] = pre_syn_anno.distance_to_root() / 1000

**Questions:**

Before running the next cell, think about the relationship between these distances.

1. Which distance should be the shortest? The longest?
2. What limitation does geodesic distance have that Euclidean and radial distance does not have?

In [ ]:
bins = np.linspace(0, 800, 20)

fig, ax = plt.subplots(figsize=(5, 5), dpi=150)

sns.histplot(
    pre_syn_df["dist_to_root_euc"],
    color=sns.color_palette("Dark2")[0],
    element="step",
    bins=bins,
    alpha=0.1,
)

sns.histplot(
    pre_syn_df["dist_to_root_radial"],
    color=sns.color_palette("Dark2")[1],
    element="step",
    bins=bins,
    alpha=0.1,
)

sns.histplot(
    pre_syn_df["dist_to_root_geo"],
    color=sns.color_palette("Dark2")[2],
    element="step",
    bins=bins,
    alpha=0.1,
)

ax.set_xlabel(r"Distance to Cell Body ($\mu$m)")
ax.legend(["Euclidean", "Radial", "Geodesic"])

**Question:**

Does the relative distributions of these curves make sense? 

#### How much axon is there?

The above just looked at synapses. If you want to know about the same distribution for axon itself, we can use similar functions on the skeleton to get this information:
1) `skeleton.distance_to_root` gives distance to root for each vertex (in nm by default)
2) `skeleton.half_edge_length` gives the amount of branch length associated with each skeleton vertex.

In [ ]:
dist_axon_um = cell_axon.skeleton.distance_to_root() / 1_000
len_axon_um = cell_axon.skeleton.half_edge_length / 1_000

In [ ]:
# For a quick plot, we can just treat this as a length-weighted histogram of skeleton vertex distances.

fig, ax = plt.subplots(figsize=(5, 5), dpi=150)

sns.histplot(
    x=dist_axon_um,
    weights=len_axon_um,
    binwidth=10,
    color="lightgray",
    element="step",
    ax=ax,
)

sns.histplot(
    pre_syn_df["dist_to_root_geo"],
    color=sns.color_palette("Dark2")[2],
    element="step",
    bins=bins,
    alpha=1,
    fill=False,
    linewidth=4,
)

ax.set_ylabel("Microns of axon")

Similar approaches to synapses could be used to find radial and euclidean distances as well, using `cell.skeleton.vertices` to get the location of each skeleton vertex. 

## Cell Types and Distance

Now we come to our central question: how does target cell type vary with synapse distance?

In [ ]:
# What is the distance between the median distance for excitatory and inhibitory targets?

pre_syn_df.groupby("synapse_target_class").agg(
    med_dist_euc=pd.NamedAgg("dist_to_root_euc", "median"),
    med_dist_rad=pd.NamedAgg("dist_to_root_radial", "median"),
    med_dist_geo=pd.NamedAgg("dist_to_root_geo", "median"),
)

In [ ]:
# Set some configuration variables for what's to come.

inh_color = "tomato"
exc_color = "navy"

distance_column = "dist_to_root_geo"

Let's visualize where in space all the synapses are as a function of distance and depth in cortex

In [ ]:
# Extract the depth of each synapse
pre_syn_df["syn_depth_um"] = v1dd_ds.transform("nm").apply_dataframe(
    "ctr_pt_position", pre_syn_df, projection="y"
)

# Restrict to only those cells with a target
pre_syn_df_ct = pre_syn_df.dropna(subset="synapse_target_class")

fig, ax = plt.subplots(figsize=(5, 4), dpi=150)

sns.scatterplot(
    y="syn_depth_um",
    x=distance_column,
    data=pre_syn_df_ct,
    hue="synapse_target_class",
    palette={"excitatory": exc_color, "inhibitory": inh_color},
    ax=ax,
    s=8,
)

ax.invert_yaxis()
ax.set_aspect("equal")
ax.legend().set_bbox_to_anchor((1, 1))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2), dpi=150)
sns.stripplot(
    x=distance_column,
    y="synapse_target_class",
    data=pre_syn_df_ct,
    hue="synapse_target_class",
    palette={"inhibitory": inh_color, "excitatory": exc_color},
    s=3,
    alpha=0.5,
)

sns.pointplot(
    x=distance_column,
    y="synapse_target_class",
    data=pre_syn_df_ct,
    color="k",
    linestyle="none",
)


In [ ]:
sns.histplot(
    x=distance_column,
    hue="synapse_target_class",
    multiple="stack",
    data=pre_syn_df,
    palette={"excitatory": exc_color, "inhibitory": inh_color},
    hue_order=["excitatory", "inhibitory"],
)

### Approach 1: Binned inhibitory targeting fraction

Our first approach will be to split synapses into distance bins.
For each bin, we will treat the fraction of inhibitory targets as arising from a binomial process and estimate the probability.

In [ ]:
# 10 bins along the first 300 microns of synapse-rich axon.

bins = np.linspace(
    pre_syn_df_ct[distance_column].min(),
    pre_syn_df_ct[distance_column].min() + 401,
    10,
)

In [ ]:
ds = pre_syn_df_ct[distance_column]
is_inhib = pre_syn_df_ct["synapse_target_class"] == "inhibitory"

k, _, _ = stats.binned_statistic(ds, is_inhib, "sum", bins=bins)
n, _, _ = stats.binned_statistic(ds, is_inhib, "count", bins=bins)

binom_res_inh = stats.binomtest(k=k, n=n)

# Handy for plots

bin_centers = bins[:-1] + np.diff(bins) / 2
binom_res_exc = stats.binomtest(k=n - k, n=n)

In [ ]:
# Make a function that plots the estimated probability and confidence interval from a scipy binomial test


def plot_binom_filled(
    x,
    binom_res,
    color=None,
    linewidth=1,
    fill_alpha=0.25,
    method="wilson",
    ax=None,
    line_kwargs=None,
    fill_kwargs=None,
):
    """
    Parameters
    ----------
    x: np.ndarray
        X-axis locations associated with the binomial test
    binom_res:
        Results of scipy.stats.binomtest
    color: list or str
        Matplotlib-interpretable color
    linewidth:
        Width of the estimate line
    fill_alpha:
        Opacity of the confidence interval fill
    method:
        Method for confidence interal, by default "wilson"
    ax:
        matplotlib axis
    line_kwargs:
        Additional dictionary of keywords for the estimate plot (see ax.plot)
    fill_kwarg:
        Additional dictionary of keywords for the confidence interval plot (see ax.fill_between)
    """

    if ax is None:
        ax = plt.gca()
    if line_kwargs is None:
        line_kwargs = {}
    if fill_kwargs is None:
        fill_kwargs = {}

    ax.fill_between(
        x,
        binom_res.proportion_ci(method=method).low,
        binom_res.proportion_ci(method=method).high,
        color=color,
        alpha=fill_alpha,
        **fill_kwargs,
    )
    ax.plot(
        x,
        binom_res.proportion_estimate,
        color=color,
        linewidth=linewidth,
        **line_kwargs,
    )
    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3), dpi=150)

plot_binom_filled(
    bin_centers,
    binom_res_exc,
    ax=ax,
    color=exc_color,
)

plot_binom_filled(
    bin_centers,
    binom_res_inh,
    ax=ax,
    color=inh_color,
)

**Questions:**

1) What are the weaknesses and assumptions of this approach?
2) Look at other cells. How typical is this range?
3) What do you notice about the typical pattern of where synapses are and where the crossover between inhibitory and excitatory targets is?
4) If you have time, use the cell type field (not just "classification") to explore if any specific inhibitory subclass drives this effect?

### Approach 2: Fit all the data together

It looks like the probability of a synapse being inhibitory is monotonic, with a value generally decreasing and stabilizing to some steady value with distance.


We can model this by fitting all the points to a sigmoidal curve, which is a generic S-like shape that smoothly steps from one value to another.

Unlike the binned data, a fitting-based approach accounts for both the exact distances and the amount of data available along the range of distances.

To start, we need a way to parameterize the sigmoidal function. A typical approach is as follows:

If we assume that the probability of a synapse being inhibitory at a given distance follows a sigmoidal function, there are two parameters to fit, which we call $\beta_0$ and $\beta_1$.
$$p(s_{inhib} \mid d) = \frac{(p_{i}-p_{lo})}{1 + e^{-(\beta_0 + \beta_1 d)}} + p_{lo}$$

The values all have meanings. $p_{lo}$ is the probability at very low values of distance, $p_{hi}$ is the probability at high values of distance, $\beta_0$ sets where the crossover happens while $\beta_1$ says how fast the crossover happens, and in which direction.
Negative values mean the probability decreases with distance, positive values indicate it increases with distance.

A standard approach to fitting data is figure out what parameters $\beta_0$ and $\beta_1$ maximum the probability of the data points.
This is *maximum likelihood estimatation*.
In practice, we typically minimize the negative of the log of the probability, simply because that is an easier number to work with and has the same answer.

In [ ]:
from scipy.special import expit  # Numerically stable version of 1 / (1 + exp(-z))
from scipy.optimize import minimize


def sigmoid_prob_is_inhib(beta, ds):
    """beta = [beta_0, beta_1, p_lo, p_hi]

    p_lo, p_hi are the asymptotes of the sigmoid: p -> p_hi in the direction
    of increasing (beta_0 + beta_1 * ds), p -> p_lo in the other.

    With beta_1 < 0 (as in the fit above), increasing distance means
    decreasing z, so:
        p_lo = p(inhib | ds -> +inf)   <- the distal floor, your A_inf
        p_hi = p(inhib | ds -> -inf)   <- the proximal ceiling
    """
    b0, b1, p_lo, p_hi = beta
    return p_lo + (p_hi - p_lo) * expit(b0 + b1 * ds)


def sigmoid_log_loss(beta, ds, value):
    """Negative log likelihood"""
    p = sigmoid_prob_is_inhib(beta, ds)
    p = np.clip(
        p, 1e-12, 1 - 1e-12
    )  # Don't let values get to 0/1 if p_lo == 0 or p_hi == 1
    return -np.sum(value * np.log(p) + (1 - value) * np.log(1 - p))


In [ ]:
## Do the fit
distance_column = "dist_to_root_geo"
ds = pre_syn_df_ct[distance_column]
is_inhib = pre_syn_df_ct["synapse_target_class"] == "inhibitory"

fit = minimize(
    sigmoid_log_loss,
    [0, 0, 0.05, 0.95],  # Initial guesses
    args=(ds, is_inhib),  # Data
)

## Plot the results

x_grid = np.arange(
    pre_syn_df_ct[distance_column].min(), pre_syn_df_ct[distance_column].max(), 10
)

fig, ax = plt.subplots(figsize=(4, 3), dpi=150)
ax.plot(x_grid, sigmoid_prob_is_inhib(fit.x, x_grid), color=inh_color)

rng = np.random.default_rng(0)
jit = rng.uniform(-0.04, 0.04, len(is_inhib))
ax.scatter(
    ds,
    is_inhib + jit,
    s=2,
    alpha=0.5,
    c=np.where(is_inhib, inh_color, exc_color),
)
ax.set_xlabel(distance_column)

**Questions**

1. Does this hold with other cells?
2. Consider both the trends and the absolute numbers. What assumptions are required for this analysis to be correct? Do you think these hold in practice?
3. What about inhibitory cells?
4. How do all of these functions behave if we chose different distance functions? What do you think that means biologically?
5. Can you think of any other ways to split up the connections for these cells?
6. This sigmoidal fit was just one model of how connections form and are distributed across an arbor. What aspects of the data does it capture, and which does it not?
7. Can you think of different models that might approach this data differently?

#### Sanity checking the results by looking at the data

In [ ]:
from nglui.statebuilder import ViewerState

We can start with a very basic neuroglancer link to the dataset based on the info stored in the CAVEclient

In [ ]:
viewer = ViewerState(client=client).add_layers_from_client()
viewer.to_link()

In [ ]:
(
    ViewerState(client=client)
    .add_layers_from_client()
    .add_segments(
        [root_id], segment_colors="white"
    )  # Add our root id and set the color to white
    .add_annotation_layer(
        "inh_targeting",  # Add a new annotation layer to put inhibitory-targeting points.
        linked_segmentation="segmentation",  # Attach it to the layer "segmentation" so that segment ids can be loaded
        color=inh_color,  # Set the base color of annotations
    )
    .add_points(  # Add points based on a dataframe
        data=pre_syn_df_ct.query('synapse_target_class == "inhibitory"'),
        name="inh_targeting",  # Add points to the layer above, which was named "inh_targeting"
        point_column="post_pt_position",  # Use the `post_pt_position` column to find locations to add points
        segment_column="post_pt_root_id",  # Use the `post_pt_root_id` column to associate a target segment with the point
        data_resolution=[
            1,
            1,
            1,
        ],  # Note that points are in nm, allowing the tool to put them in the right location in neuroglancer space
    )
    .add_annotation_layer(  # Do the same thing again for excitatory-targeting synapses
        "exc_targeting",
        linked_segmentation="segmentation",
        color="cyan",
    )
    .add_points(
        data=pre_syn_df_ct.query('synapse_target_class == "excitatory"'),
        name="exc_targeting",
        point_column="post_pt_position",
        segment_column="post_pt_root_id",
        data_resolution=[1, 1, 1],
    )
).to_clipboard()

## Additional Ideas

1) The Schmidt et al. paper measures a few other aspects of neuronal connectivity, such as the clustering of synapses in the same connection and the connectivity of the target neuron. Read that paper and think about how its approach might relate to the EM datasets.

2) Another interesting kind of distance is [Electrotonic Distance](https://pages.ucsd.edu/~msereno/systneurosci/readings/01.00-DendriteSpikeSyn.pdf), which measures how much a voltage signal attenuates across a neuronal arbor with passive ion channels. Electrotonic distance incorporates dendritic radius, branching structure, and membrane properties in a manner that allows one to estimate dendritic integration. On a similar vein, [Morabito et al. 2025](https://www.cell.com/neuron/fulltext/S0896-6273(25)00429-5?_returnURL=https%3A%2F%2Flinkinghub.elsevier.com%2Fretrieve%2Fpii%2FS0896627325004295%3Fshowall%3Dtrue) used the MICrONS dataset to identify some interesting differences in dendritic integration propertie of inhibitory subclasses.

3) How do different neurons differ in how they connect to their targets? For example, do they make repeated sets of synapses onto the same target, or distribute them more randomly? What might different rules mean for circuit structure? 